(phase 1) I have to move the related pdbqt files from the source folder to these folders. Then (phase 2) I can perform the docking in each folder separately, (phase 3) add these new affinity data to my csv file, and (phase 4) analyze the related metrics of their similarity to Predicted Affinity.

In [2]:
from pathlib import Path  # Portable paths
import numpy as np        # Simple math
import pandas as pd       # CSV I/O and sampling

In [3]:
# ============================================
# Phase 1 (Windows) — Build in_AD and out_AD filename lists
# Uses explicit columns: number, pre_reg_affinity, inAD_final
# - in_AD: stratified random sample (by pre_reg_affinity quantiles) of 300
# - out_AD: first 300 by ascending pre_reg_affinity
# - Writes text lists with exact filenames: drug{number}.pdbqt
# ============================================

from pathlib import Path  # Portable paths
import numpy as np        # Simple math
import pandas as pd       # CSV I/O and sampling

# ------------- Configuration -------------
BASE = Path(r"C:\CKI\EV")  # Change to your Windows folder
CSV_PATH = BASE / "df_full_wo_features.csv"              # Your small CSV with required columns
OUT_DIR = BASE                                           # Where to write the two lists

# Required column names (as stated)
NUMBER_COL = "number"                # Integer ID used in filenames
AFF_COL = "pre_reg_affinity"         # Numeric affinity (more negative = stronger)
INAD_COL = "inAD_final"              # Boolean-like membership

# Selection sizes and sampling details
IN_AD_N = 300                        # How many in-AD
OUT_AD_N = 300                       # How many out-of-AD
Q_BINS = 10                          # Quantile bins for stratification of in-AD
RANDOM_SEED = 42                     # Reproducibility
PAD_WIDTH = 0                        # 0 = no zero-padding; set e.g. 6 for drug000123.pdbqt

# ------------- Load and normalize -------------
df = pd.read_csv(CSV_PATH)  # Read CSV with pandas robust parser
# Normalize inAD_final to boolean
inad_bool = df[INAD_COL].astype(str).str.strip().str.lower().isin(["true", "1", "yes", "t", "y"])
df = df.assign(inad_bool=inad_bool)

# Ensure required columns exist and types are correct
df = df[pd.notnull(df[NUMBER_COL]) & pd.notnull(df[AFF_COL])].copy()
df[NUMBER_COL] = df[NUMBER_COL].astype(int)
df[AFF_COL] = pd.to_numeric(df[AFF_COL], errors="coerce")
df = df[pd.notnull(df[AFF_COL])].copy()

# ------------- Helper: make filename -------------
def make_name(n: int) -> str:
    # Build "drug{number}.pdbqt" with optional zero-padding
    return f"drug{int(n):0{PAD_WIDTH}d}.pdbqt" if PAD_WIDTH > 0 else f"drug{int(n)}.pdbqt"

# ------------- Stratified sample for in-AD -------------
def stratified_sample_by_quantiles(df_in: pd.DataFrame, value_col: str, total_n: int, q_bins: int, seed: int) -> pd.DataFrame:
    # Bin by quantiles; fallback to equal-width bins if qcut fails on ties
    try:
        bins = pd.qcut(df_in[value_col], q=q_bins, labels=False, duplicates="drop")
    except Exception:
        bins = pd.cut(df_in[value_col], bins=q_bins, labels=False, include_lowest=True)
    dfb = df_in.assign(_bin=bins)

    # Count per bin and allocate samples proportionally (rounded, with remainder distributed)
    counts = dfb["_bin"].value_counts(dropna=False).sort_index()
    props = counts / counts.sum()
    raw = props * total_n
    base = np.floor(raw).astype(int)
    remainder = total_n - base.sum()
    # Distribute remainder to bins with largest fractional parts
    frac = (raw - base).sort_values(ascending=False)
    for b in frac.index[:max(remainder, 0)]:
        base.loc[b] += 1

    # Sample per bin (cap at bin size)
    parts = []
    rng = np.random.RandomState(seed)
    for b, n_take in base.items():
        grp = dfb[dfb["_bin"] == b]
        if len(grp) == 0 or n_take <= 0:
            continue
        n_take = min(n_take, len(grp))
        parts.append(grp.sample(n=n_take, random_state=rng))

    if parts:
        return pd.concat(parts, axis=0).drop(columns=["_bin"])
    else:
        return df_in.head(0)

# Subsets
df_in = df[df["inad_bool"] == True].copy()
df_out = df[df["inad_bool"] == False].copy()

# Stratified 300 in-AD by pre_reg_affinity distribution
n_in = min(IN_AD_N, len(df_in))
sel_in = stratified_sample_by_quantiles(df_in, AFF_COL, n_in, Q_BINS, RANDOM_SEED)

# First 300 out-of-AD by ascending affinity (more negative first)
n_out = min(OUT_AD_N, len(df_out))
sel_out = df_out.sort_values(by=AFF_COL, ascending=True).head(n_out)

# ------------- Build filename lists -------------
in_files = [make_name(n) for n in sel_in[NUMBER_COL].tolist()]
out_files = [make_name(n) for n in sel_out[NUMBER_COL].tolist()]

# ------------- Write lists (LF newlines) -------------
(OUT_DIR / "in_ad_files.txt").write_text("\n".join(in_files) + "\n", encoding="utf-8")
(OUT_DIR / "out_ad_files.txt").write_text("\n".join(out_files) + "\n", encoding="utf-8")

# Optional: write selection CSVs for traceability
sel_in.assign(filename=in_files).to_csv(OUT_DIR / "in_ad_list.csv", index=False)
sel_out.assign(filename=out_files).to_csv(OUT_DIR / "out_ad_list.csv", index=False)

# ------------- Console hints -------------
print(f"Wrote lists to: {OUT_DIR}")
print(f"in-AD rows: {len(sel_in)}; out-AD rows: {len(sel_out)}")
print("Example in-AD filenames:", in_files[:5])
print("Example out-AD filenames:", out_files[:5])


Wrote lists to: C:\CKI\EV
in-AD rows: 300; out-AD rows: 300
Example in-AD filenames: ['drug78071.pdbqt', 'drug192996.pdbqt', 'drug344148.pdbqt', 'drug371748.pdbqt', 'drug85154.pdbqt']
Example out-AD filenames: ['drug697062.pdbqt', 'drug110875.pdbqt', 'drug56916.pdbqt', 'drug56914.pdbqt', 'drug56915.pdbqt']


In [1]:
# ============================================
# Phase 3 — Integrate docking affinities into the master CSV
# Files expected in current working directory:
#   - df_full_wo_features.csv
#   - in_ad_Summary_Final.txt
#   - out_ad_Summary_Final.txt
# Output:
#   - df_full_wo_features_with_ev.csv (updated table)
#   - ev_affinity_map.csv (number, ev_true_affinity)
#   - merge_report.txt (match/miss counts)
# ============================================

# -------- Imports --------
from pathlib import Path  # File paths
import re                 # Line parsing
import pandas as pd       # CSV I/O and joins

# -------- Configuration --------
BASE = Path(".")  # Current directory
CSV_IN = BASE / "df_full_wo_features.csv"  # Main table
IN_SUM = BASE / "in_ad_Summary_Final.txt"  # in-AD docking results
OUT_SUM = BASE / "out_ad_Summary_Final.txt"  # out-AD docking results
CSV_OUT = BASE / "df_full_wo_features_with_ev.csv"  # Updated table
MAP_OUT = BASE / "ev_affinity_map.csv"  # Number-to-affinity mapping
REPORT = BASE / "merge_report.txt"  # Small text report

# -------- Helpers --------
def parse_summary_file(path: Path, tag: str):
    """
    Parse a Summary_Final.txt-like file with lines such as:
    drug88457.pdbqt.txt   -8.6    drug88457.pdbqt.txt
    Returns a list of dicts with fields: number (int), affinity (float), set (tag).
    """
    rows = []  # Accumulate parsed rows
    if not path.exists():
        return rows
    with path.open("r", encoding="utf-8", errors="ignore") as fh:
        for line in fh:
            line = line.strip()
            if not line:
                continue
            # Split on whitespace; expected: [name, score, name]
            parts = line.split()
            if len(parts) < 2:
                continue
            name = parts[0]
            score_txt = parts[1]
            # Extract integer from "drug12345.pdbqt.txt"
            m = re.search(r"drug(\d+)\.pdbqt\.txt", name, flags=re.IGNORECASE)
            if not m:
                continue
            try:
                number = int(m.group(1))
            except Exception:
                continue
            # Parse affinity (kcal/mol)
            try:
                affinity = float(score_txt)
            except Exception:
                # Fallback: search any float in the line
                mf = re.search(r"(-?\d+(?:\.\d+)?)", line)
                if not mf:
                    continue
                affinity = float(mf.group(1))
            rows.append({"number": number, "affinity": affinity, "set": tag})
    return rows

# -------- Read and prepare inputs --------
df = pd.read_csv(CSV_IN)  # Load master table [pandas read_csv] [web:140][web:146]
# Ensure "number" is clean integer for merging
if "number" not in df.columns:
    raise ValueError('Required column "number" not found in df_full_wo_features.csv')
df = df[pd.notnull(df["number"])].copy()
df["number"] = df["number"].astype(int)

# Parse both EV summary files
rows_in = parse_summary_file(IN_SUM, tag="in_ad")
rows_out = parse_summary_file(OUT_SUM, tag="out_ad")
rows_all = rows_in + rows_out

# Build a tidy mapping (if duplicates, keep the first occurrence)
map_df = pd.DataFrame(rows_all)
if not map_df.empty:
    # If a number appears multiple times, keep the first (files are usually sorted best-to-worst)
    map_df = map_df.sort_values(by=["number"]).drop_duplicates(subset=["number"], keep="first")
    map_df = map_df[["number", "affinity"]].rename(columns={"affinity": "ev_true_affinity"})
else:
    # Create empty structure if no rows parsed
    map_df = pd.DataFrame(columns=["number", "ev_true_affinity"])

# -------- Merge and update columns --------
# Create ev_true_affinity if missing
if "ev_true_affinity" not in df.columns:
    df["ev_true_affinity"] = pd.NA

# Create dock_true_affinity if missing
if "dock_true_affinity" not in df.columns:
    df["dock_true_affinity"] = pd.NA

# Merge EV affinities by "number"
df = df.merge(map_df, on="number", how="left", suffixes=("", "_new"))  # Left join [web:140][web:146]

# Update ev_true_affinity with newly merged values
if "ev_true_affinity_new" in df.columns:
    df["ev_true_affinity"] = df["ev_true_affinity_new"].combine_first(df["ev_true_affinity"])
    df = df.drop(columns=["ev_true_affinity_new"])

# Update dock_true_affinity only where it is missing (preserve prior labels if present)
df["dock_true_affinity"] = df["dock_true_affinity"].combine_first(df["ev_true_affinity"])

# -------- Write outputs --------
# Save updated master CSV
df.to_csv(CSV_OUT, index=False)  # Write updated table [web:140][web:146]

# Save mapping for audit
map_df.to_csv(MAP_OUT, index=False)  # number -> ev_true_affinity [web:140][web:146]

# Simple merge report
matched = map_df["number"].nunique()
total_candidates = df["number"].nunique()
with REPORT.open("w", encoding="utf-8") as r:
    r.write(f"EV mapping rows parsed: {len(rows_all)}\n")
    r.write(f"Unique numbers with EV affinities: {matched}\n")
    r.write(f"Unique numbers in master CSV: {total_candidates}\n")

print(f"Updated table: {CSV_OUT.resolve()}")       # Path to updated CSV [web:140][web:146]
print(f"Affinity map  : {MAP_OUT.resolve()}")      # Path to mapping CSV [web:140][web:146]
print(f"Report        : {REPORT.resolve()}")       # Path to merge report [web:140][web:146]


Updated table: C:\CKI\EV\df_full_wo_features_with_ev.csv
Affinity map  : C:\CKI\EV\ev_affinity_map.csv
Report        : C:\CKI\EV\merge_report.txt


In [12]:
# ============================================
# Phase 3 (robust) — Integrate docking affinities only when a valid score is present
# - Reads: df_full_wo_features.csv, in_ad_Summary_Final.txt, out_ad_Summary_Final.txt
# - Strictly parses lines like: "drug123.pdbqt.txt   -8.6   drug123.pdbqt.txt"
# - Accepts affinity only if the middle token is a real float AND in a plausible Vina range [-50, 0]
# - Leaves ev_true_affinity as NaN when no valid affinity is found (no fallback to any number)
# - Updates dock_true_affinity only where missing and a valid EV value exists
# - Writes: df_full_wo_features_with_ev_fixed.csv and ev_affinity_map_fixed.csv
# ============================================

from pathlib import Path  # Path handling
import re                 # Regex parsing
import pandas as pd       # CSV I/O and merging
import numpy as np        # NaN handling

# ---------------------------
# Paths (run in the directory containing the files)
# ---------------------------
BASE = Path(".")  # current directory
CSV_IN  = BASE / "df_full_wo_features.csv"                 # input master CSV
SUM_IN  = BASE / "in_ad_Summary_Final.txt"                 # in-AD summary
SUM_OUT = BASE / "out_ad_Summary_Final.txt"                # out-AD summary
CSV_OUT = BASE / "df_full_wo_features_with_ev_fixed.csv"   # output master CSV
MAP_OUT = BASE / "ev_affinity_map_fixed.csv"               # mapping for audit

# ---------------------------
# Strict parser for Summary_Final lines
# ---------------------------
# Pattern: filename, affinity, filename (same), separated by whitespace
# Example: "drug88457.pdbqt.txt   -8.6   drug88457.pdbqt.txt"
LINE_RE = re.compile(
    r'^(drug(\d+)\.pdbqt\.txt)\s+([-+]?\d+(?:\.\d+)?)\s+\1\s*$',
    flags=re.IGNORECASE,
)

def parse_summary_file(path: Path, tag: str):
    """Return list of dicts: {'number': int, 'affinity': float, 'set': tag} for valid lines only."""
    rows = []  # accumulate parsed rows
    if not path.exists():
        return rows  # silently handle missing file
    # Read lines with tolerant decoding
    for line in path.read_text(encoding="utf-8", errors="ignore").splitlines():
        line = line.strip()
        if not line:
            continue  # skip blanks
        m = LINE_RE.match(line)
        if not m:
            continue  # skip malformed lines
        num_str = m.group(2)  # digits after 'drug'
        aff_str = m.group(3)  # affinity as string
        try:
            n = int(num_str)  # ligand number
            aff = float(aff_str)  # affinity value
        except Exception:
            continue  # skip if conversion fails
        # Enforce plausible Vina range to avoid accidental parsing of IDs as scores
        if not (-50.0 <= aff <= 0.0):
            continue  # invalid score -> treat as absent
        rows.append({"number": n, "affinity": aff, "set": tag})
    return rows  # only valid entries included

# ---------------------------
# Build the number -> affinity map from both summaries
# ---------------------------
rows_in  = parse_summary_file(SUM_IN,  tag="in_ad")   # parse in-AD
rows_out = parse_summary_file(SUM_OUT, tag="out_ad")  # parse out-AD
rows_all = rows_in + rows_out                         # union of parsed rows

# Create mapping DataFrame (unique by number)
if rows_all:
    map_df = pd.DataFrame(rows_all)                              # to DataFrame
    map_df = map_df.sort_values(["number", "set"]).drop_duplicates("number", keep="first")  # one per number
    map_df = map_df[["number", "affinity"]].rename(columns={"affinity": "ev_true_affinity"})  # rename column
else:
    map_df = pd.DataFrame(columns=["number", "ev_true_affinity"])  # empty mapping

# ---------------------------
# Load master CSV and normalize key types
# ---------------------------
df = pd.read_csv(CSV_IN)                 # read master table
if "number" not in df.columns:
    raise ValueError('Required column "number" not found in df_full_wo_features.csv')  # ensure key exists
df = df[pd.notnull(df["number"])].copy()  # keep rows with a number
df["number"] = df["number"].astype(int)   # integer key

# Ensure target columns exist
if "ev_true_affinity" not in df.columns:
    df["ev_true_affinity"] = np.nan       # create if missing
if "dock_true_affinity" not in df.columns:
    df["dock_true_affinity"] = np.nan     # create if missing

# ---------------------------
# Left-join mapping; leave NaN when no valid affinity is found
# ---------------------------
df = df.merge(map_df, on="number", how="left", suffixes=("", "_new"))  # left merge adds ev_true_affinity_new

# Overwrite ev_true_affinity only where a new valid value exists
if "ev_true_affinity_new" in df.columns:
    df["ev_true_affinity"] = df["ev_true_affinity"].where(
        df["ev_true_affinity"].notna(), df["ev_true_affinity_new"]
    )  # fill only missing with new values
    df = df.drop(columns=["ev_true_affinity_new"])  # drop temp column

# Fill dock_true_affinity only where missing and ev_true_affinity is available
df["dock_true_affinity"] = df["dock_true_affinity"].where(
    df["dock_true_affinity"].notna(), df["ev_true_affinity"]
)  # do not overwrite existing labels

# ---------------------------
# Save outputs and small console report
# ---------------------------
df.to_csv(CSV_OUT, index=False)         # write updated master CSV
map_df.to_csv(MAP_OUT, index=False)     # write mapping for audit

# Console summary
print(f"Parsed from in_ad:  {len(rows_in)} valid rows")     # count in-AD rows
print(f"Parsed from out_ad: {len(rows_out)} valid rows")    # count out-AD rows
print(f"Unique mapped IDs:  {map_df['number'].nunique()}")  # unique numbers with affinities
print(f"Wrote: {CSV_OUT.resolve()}")                        # output path
print(f"Wrote: {MAP_OUT.resolve()}")                        # mapping path

Parsed from in_ad:  299 valid rows
Parsed from out_ad: 295 valid rows
Unique mapped IDs:  594
Wrote: C:\CKI\EV\df_full_wo_features_with_ev_fixed.csv
Wrote: C:\CKI\EV\ev_affinity_map_fixed.csv


In [13]:
df.tail(10)

,number,SMILES,Name,dock_true_affinity,pre_reg_affinity,pre_cls_activity,inAD_final,ev_true_affinity
781169,782188,O=S1(=O)C[C@H](O)[C@@H](S(=O)(=O)Cl)C1,ZINC000100004586,NaN,-5.182879,0,False,NaN
781170,782189,O=S1(=O)C[C@@H](O)[C@@H](S(=O)(=O)Cl)C1,ZINC000100004589,NaN,-5.182879,0,False,NaN
781171,782190,COC[C@](C)(O)CNC(=O)c1cn(C)nn1,ZINC000217570620,NaN,-5.410433,0,True,NaN
781172,782191,COC[C@@](C)(O)CNC(=O)c1cn(C)nn1,ZINC000217570706,NaN,-5.410433,0,True,NaN
781173,782192,O=C(CO)N[C@@H]1CCOC[C@H]1OCCO,ZINC000218243357,NaN,-5.034461,0,True,NaN
781174,782193,COC[C@@H]1[C@H](NC(CO)CO)[C@@H]2CCO[C@H]12,ZINC000218404185,NaN,-5.147846,0,True,NaN
781175,782194,CN(C)CCO[C@@H]1COCC[C@H]1NC(=O)CO,ZINC000218743468,NaN,-5.087768,0,True,NaN
781176,782195,Cn1cc(S(=O)(=O)F)c(=O)n(C)c1=O,ZINC000238857165,NaN,-5.307266,0,False,NaN
781177,782196,O=C1NC(=O)[C@@H](CCS(=O)(=O)F)N1,ZINC000307689379,NaN,-5.328358,0,True,NaN
781178,782197,O=C1NC(=O)[C@H](CCS(=O)(=O)F)N1,ZINC000307689380,NaN,-5.328358,0,True,NaN


In [ ]:
# ============================================
# Phase 4 — External validation metrics for ev_true_affinity vs pre_reg_affinity
# - Computes metrics separately for inAD_final = True and False
# - Metrics: N, Pearson r, r^2, Spearman ρ, MAE, RMSE, CCC, bias, LoA, slope, intercept
# - Writes per-subset CSVs and a combined CSV
# ============================================

# ---- Imports ----
import math  # Basic math
from pathlib import Path  # Paths
import numpy as np  # Arrays and stats
import pandas as pd  # Data handling

# ---- Configuration ----
BASE = Path(".")  # Current directory
CSV_IN = BASE / "df_full_wo_features_with_ev.csv"  # Input with ev_true_affinity
CSV_OUT_COMBINED = BASE / "ev_metrics_by_AD.csv"  # Combined metrics
CSV_OUT_IN = BASE / "ev_metrics_inAD.csv"  # In-AD metrics only
CSV_OUT_OUT = BASE / "ev_metrics_outAD.csv"  # Out-of-AD metrics only

# Column names
COL_TRUE = "ev_true_affinity"         # Observed from docking (phase 3)
COL_PRED = "pre_reg_affinity"         # Model prediction
COL_AD   = "inAD_final"               # Applicability domain flag

# ---- Helpers ----
def _nan_filter(a, b):
    """Return arrays with matching non-NaN entries."""
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    mask = np.isfinite(a) & np.isfinite(b)
    return a[mask], b[mask]

def pearson_r(y_true, y_pred):
    """Pearson correlation."""
    y_true, y_pred = _nan_filter(y_true, y_pred)
    if y_true.size < 2:
        return np.nan
    r = np.corrcoef(y_true, y_pred)[0, 1]
    return float(r)

def spearman_rho(y_true, y_pred):
    """Spearman correlation via ranking + Pearson on ranks."""
    y_true, y_pred = _nan_filter(y_true, y_pred)
    if y_true.size < 2:
        return np.nan
    # Rank average for ties
    rt = pd.Series(y_true).rank(method="average").to_numpy()
    rp = pd.Series(y_pred).rank(method="average").to_numpy()
    rho = np.corrcoef(rt, rp)[0, 1]
    return float(rho)

def mae(y_true, y_pred):
    y_true, y_pred = _nan_filter(y_true, y_pred)
    if y_true.size == 0:
        return np.nan
    return float(np.mean(np.abs(y_pred - y_true)))

def rmse(y_true, y_pred):
    y_true, y_pred = _nan_filter(y_true, y_pred)
    if y_true.size == 0:
        return np.nan
    return float(np.sqrt(np.mean((y_pred - y_true) ** 2)))

def ccc(y_true, y_pred):
    """Lin's Concordance Correlation Coefficient."""
    y_true, y_pred = _nan_filter(y_true, y_pred)
    n = y_true.size
    if n < 2:
        return np.nan
    mu_x = np.mean(y_pred)
    mu_y = np.mean(y_true)
    var_x = np.var(y_pred, ddof=0)
    var_y = np.var(y_true, ddof=0)
    cov_xy = np.mean((y_pred - mu_x) * (y_true - mu_y))
    denom = var_x + var_y + (mu_x - mu_y) ** 2
    if denom == 0:
        return np.nan
    return float((2 * cov_xy) / denom)

def calibration(y_true, y_pred):
    """Slope and intercept from regressing y_true on y_pred."""
    y_true, y_pred = _nan_filter(y_true, y_pred)
    if y_true.size < 2:
        return (np.nan, np.nan)
    slope, intercept = np.polyfit(y_pred, y_true, 1)
    return float(slope), float(intercept)

def bland_altman(y_true, y_pred):
    """Bias and 95% limits of agreement for diff = pred - true."""
    y_true, y_pred = _nan_filter(y_true, y_pred)
    if y_true.size == 0:
        return (np.nan, np.nan, np.nan)
    diff = y_pred - y_true
    bias = float(np.mean(diff))
    sd = float(np.std(diff, ddof=1)) if diff.size > 1 else 0.0
    loa_low = bias - 1.96 * sd
    loa_high = bias + 1.96 * sd
    return bias, loa_low, loa_high

def compute_metrics(df):
    """Compute all metrics for rows with valid values."""
    y_true = pd.to_numeric(df[COL_TRUE], errors="coerce").to_numpy()
    y_pred = pd.to_numeric(df[COL_PRED], errors="coerce").to_numpy()
    y_true, y_pred = _nan_filter(y_true, y_pred)
    n = y_true.size
    if n == 0:
        return {
            "N": 0, "pearson_r": np.nan, "r2": np.nan, "spearman_rho": np.nan,
            "MAE": np.nan, "RMSE": np.nan, "CCC": np.nan, "cal_slope": np.nan,
            "cal_intercept": np.nan, "bias": np.nan, "loa_low": np.nan, "loa_high": np.nan
        }
    r = pearson_r(y_true, y_pred)
    rho = spearman_rho(y_true, y_pred)
    m_mae = mae(y_true, y_pred)
    m_rmse = rmse(y_true, y_pred)
    m_ccc = ccc(y_true, y_pred)
    slope, intercept = calibration(y_true, y_pred)
    bias, loa_low, loa_high = bland_altman(y_true, y_pred)
    return {
        "N": int(n),
        "pearson_r": r,
        "r2": (r ** 2) if np.isfinite(r) else np.nan,
        "spearman_rho": rho,
        "MAE": m_mae,
        "RMSE": m_rmse,
        "CCC": m_ccc,
        "cal_slope": slope,
        "cal_intercept": intercept,
        "bias": bias,
        "loa_low": loa_low,
        "loa_high": loa_high,
    }

# ---- Load data and normalize ----
df = pd.read_csv(CSV_IN)  # Load combined table
# Normalize boolean for inAD_final
inad = df[COL_AD].astype(str).str.strip().str.lower().isin(["true", "1", "yes", "t", "y"])
df = df.assign(inad_bool=inad)

# ---- Split and compute ----
df_in = df[df["inad_bool"] == True].copy()
df_out = df[df["inad_bool"] == False].copy()

metrics_in = compute_metrics(df_in)
metrics_out = compute_metrics(df_out)

# Build DataFrames
df_in_metrics = pd.DataFrame([metrics_in])
df_out_metrics = pd.DataFrame([metrics_out])
df_in_metrics.insert(0, "subset", "inAD_true")
df_out_metrics.insert(0, "subset", "inAD_false")

df_all = pd.concat([df_in_metrics, df_out_metrics], ignore_index=True)

# ---- Save results ----
df_all.to_csv(CSV_OUT_COMBINED, index=False)
df_in_metrics.to_csv(CSV_OUT_IN, index=False)
df_out_metrics.to_csv(CSV_OUT_OUT, index=False)

# ---- Print a concise summary ----
print("External validation metrics")
print(df_all.to_string(index=False))
print(f"\nWrote: {CSV_OUT_COMBINED}")
print(f"Wrote: {CSV_OUT_IN}")
print(f"Wrote: {CSV_OUT_OUT}")


External validation metrics
    subset   N  pearson_r       r2  spearman_rho      MAE     RMSE      CCC  cal_slope  cal_intercept      bias   loa_low  loa_high
 inAD_true 299   0.817806 0.668807      0.803541 0.222013 0.276370 0.723074   1.018448       0.260658 -0.161419 -0.601844  0.279006
inAD_false 295   0.379812 0.144257      0.275725 0.301456 0.375324 0.250271   0.866620      -0.793928 -0.139693 -0.823637  0.544252

Wrote: ev_metrics_by_AD.csv
Wrote: ev_metrics_inAD.csv
Wrote: ev_metrics_outAD.csv


### What the metrics mean  
  
Pearson r and r² evaluate linear association and explained variance; r² is commonly reported alongside correlation and error metrics for external QSAR validation.​

Spearman’s ρ measures rank agreement, providing a monotonic association metric less sensitive to nonlinear scaling than Pearson.​

MAE averages absolute errors and RMSE is the square root of mean squared errors, both in affinity units for interpretability.​

Lin’s CCC assesses both precision and accuracy relative to the identity line, making it stricter than r alone for external predictivity.​

Bland–Altman bias and limits of agreement quantify systematic offset and expected disagreement range, complementing correlation-based summaries.

External validation metrics
    subset   N  pearson_r       r2  spearman_rho      MAE     RMSE      CCC  cal_slope  cal_intercept      bias   loa_low  loa_high
 inAD_true 299   0.817806 0.668807      0.803541 0.222013 0.276370 0.723074   1.018448       0.260658 -0.161419 -0.601844  0.279006
inAD_false 295   0.379812 0.144257      0.275725 0.301456 0.375324 0.250271   0.866620      -0.793928 -0.139693 -0.823637  0.544252

Wrote: ev_metrics_by_AD.csv
Wrote: ev_metrics_inAD.csv
Wrote: ev_metrics_outAD.csv

# Error Diagnosis

In [3]:
df_meta = pd.read_csv("df_full_wo_features_with_ev.csv")

In [10]:
# === External validation diagnostics and reliability curve for regression QSAR ===[1]
# This script loads an external validation CSV with columns: y_true, y_pred, and inAD (boolean), then computes calibration, CCC, Bland–Altman, reliability curves, bootstrap CIs, and optional per‑scaffold diagnostics if 'scaffold' exists.[1]

# -----------------------------
# Imports
# -----------------------------
import numpy as np  # imports numerical computing[2]
import pandas as pd  # imports data frames[2]
from scipy import stats  # imports statistical tests and correlations[2]
from sklearn.isotonic import IsotonicRegression  # isotonic regression for monotonic calibration[3]
from sklearn.linear_model import LinearRegression  # linear regression for calibration slope/intercept[2]
import matplotlib.pyplot as plt  # plotting library[2]
import seaborn as sns  # statistical visualization[2]
rng = np.random.default_rng(42)  # reproducible RNG[2]

# -----------------------------
# Helper: Lin’s CCC
# -----------------------------
def concordance_correlation_coefficient(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    # Implements Lin’s CCC: ρ_c = 2σ_xy / (σ_x^2 + σ_y^2 + (μ_x − μ_y)^2)[4]
    x = np.asarray(y_true)  # ensure array[4]
    y = np.asarray(y_pred)  # ensure array[4]
    mx, my = x.mean(), y.mean()  # means[4]
    vx, vy = x.var(ddof=1), y.var(ddof=1)  # sample variances[4]
    sxy = np.cov(x, y, ddof=1)  # sample covariance[5][4]
    ccc = (2 * sxy) / (vx + vy + (mx - my) ** 2)  # CCC formula[4]
    return float(ccc)  # return scalar[4]

# -----------------------------
# Helper: Bland–Altman stats
# -----------------------------
def bland_altman_stats(y_true: np.ndarray, y_pred: np.ndarray):
    # Computes bias and 95% limits of agreement assuming normality of differences: mean ± 1.96*sd[6]
    diff = np.asarray(y_pred) - np.asarray(y_true)  # differences (pred − true)[6]
    bias = diff.mean()  # mean difference (bias)[6]
    sd = diff.std(ddof=1)  # sd of differences[6]
    loa_low = bias - 1.96 * sd  # lower LoA[6]
    loa_high = bias + 1.96 * sd  # upper LoA[6]
    return float(bias), float(loa_low), float(loa_high)  # tuple of stats[6]

# -----------------------------
# Helper: calibration slope/intercept
# -----------------------------
def calibration_slope_intercept(y_true: np.ndarray, y_pred: np.ndarray):
    # Fit OLS: y_true = intercept + slope * y_pred; slope<1 suggests overfitting, slope>1 underfitting in risk models[7]
    lr = LinearRegression()  # create linear regressor[2]
    yp = np.asarray(y_pred).reshape(-1, 1)  # shape for sklearn[2]
    lr.fit(yp, np.asarray(y_true))  # fit OLS[7]
    slope = float(lr.coef_)  # slope[7]
    intercept = float(lr.intercept_)  # intercept (calibration-in-the-large for slope fixed to 1 is related)[7]
    return slope, intercept  # return parameters[7]

# -----------------------------
# Helper: rank/discrimination and error metrics
# -----------------------------
def regression_metrics(y_true: np.ndarray, y_pred: np.ndarray):
    # Compute Pearson r, Spearman ρ, R^2, MAE, RMSE, and CCC for agreement[4]
    pearson_r = stats.pearsonr(y_true, y_pred).statistic  # Pearson correlation[2]
    spearman_rho = stats.spearmanr(y_true, y_pred).correlation  # Spearman correlation[2]
    r2 = pearson_r ** 2  # R^2 from Pearson for simple correlation summary[2]
    mae = float(np.mean(np.abs(y_pred - y_true)))  # mean absolute error[2]
    rmse = float(np.sqrt(np.mean((y_pred - y_true) ** 2)))  # root mean squared error[2]
    ccc = concordance_correlation_coefficient(y_true, y_pred)  # Lin’s CCC[4]
    return pearson_r, r2, spearman_rho, mae, rmse, ccc  # pack metrics[4]

# -----------------------------
# Helper: reliability (calibration) curve for regression
# -----------------------------
def reliability_curve(y_true: np.ndarray, y_pred: np.ndarray, n_bins: int = 10):
    # Bin by predicted values, then compute mean predicted vs mean observed per bin to assess calibration shape[8]
    order = np.argsort(y_pred)  # sort by predictions[8]
    y_true_sorted = np.asarray(y_true)[order]  # sorted true[8]
    y_pred_sorted = np.asarray(y_pred)[order]  # sorted pred[8]
    bins = np.array_split(np.arange(len(y_pred_sorted)), n_bins)  # quantile bins[8]
    bin_pred = [y_pred_sorted[idx].mean() for idx in bins]  # mean pred in bin[8]
    bin_true = [y_true_sorted[idx].mean() for idx in bins]  # mean true in bin[8]
    bin_count = [len(idx) for idx in bins]  # size per bin[8]
    ece = np.average(np.abs(np.array(bin_true) - np.array(bin_pred)), weights=np.array(bin_count) / len(y_pred_sorted))  # ECE-like gap[8]
    return np.array(bin_pred), np.array(bin_true), np.array(bin_count), float(ece)  # return arrays and ECE-like[8]

# -----------------------------
# Helper: isotonic calibration of regression mean
# -----------------------------
def fit_isotonic_calibrator(y_true: np.ndarray, y_pred: np.ndarray):
    # Fit monotonic mapping g so that calibrated = g(y_pred); preserves ranking and fixes monotone bias[3]
    iso = IsotonicRegression(out_of_bounds="clip")  # create isotonic calibrator[3]
    iso.fit(y_pred, y_true)  # fit on calibration split[3]
    return iso  # return fitted calibrator[3]

# -----------------------------
# Helper: bootstrap CCC CI
# -----------------------------
def bootstrap_ccc_ci(y_true: np.ndarray, y_pred: np.ndarray, n_boot: int = 2000, alpha: float = 0.05):
    # Nonparametric bootstrap for CCC; returns percentile CI for robustness assessment[4]
    n = len(y_true)  # sample size[4]
    idx = rng.integers(0, n, size=(n_boot, n))  # bootstrap indices[4]
    ccc_samples = []
    for i in range(n_boot):  # iterate bootstrap[4]
        xi = y_true[idx[i]]  # resampled y_true[4]
        yi = y_pred[idx[i]]  # resampled y_pred[4]
        ccc_samples.append(concordance_correlation_coefficient(xi, yi))  # CCC sample[4]
    lo = float(np.quantile(ccc_samples, alpha / 2))  # lower percentile[4]
    hi = float(np.quantile(ccc_samples, 1 - alpha / 2))  # upper percentile[4]
    return lo, hi  # CI bounds[4]

# -----------------------------
# I/O: load your external validation file
# -----------------------------
# Set your CSV path here; expected columns: y_true, y_pred, inAD (True/False), optional: scaffold[1]
INPUT_CSV = "df_full_wo_features_with_ev.csv"  # replace with your file path[1]
#df = pd.read_csv("df_full_wo_features_with_ev.csv")  # load data[2]

y_pred = df["pre_reg_affinity"].values  # extract pred[2]
y_true = df["dock_true_affinity"].values  # extract true[2]
in_ad = df["inAD_final"].values  # extract in-AD flag[2]

# Basic sanity checks for required columns[2]
required_cols = {"dock_true_affinity", "pre_reg_affinity", "inAD_final"}  # required schema[1]
missing = required_cols - set(df.columns)  # compute missing[2]
if missing:  # validate schema[2]:
    raise ValueError(f"Missing required columns: {missing}")  # error if missing[2]

# -----------------------------
# Split by AD
# -----------------------------
in_ad = df[df["inAD_final"].astype(bool)].copy()  # subset in-AD[1]
out_ad = df[~df["inAD_final"].astype(bool)].copy()  # subset out-AD[1]

# -----------------------------
# Compute metrics and calibration stats
# -----------------------------
def summarize_subset(name: str, data: pd.DataFrame):
    # Compute metrics, calibration, Bland–Altman, and bootstrap CI for CCC for a subset[4]
    y_true = data["y_true"].values  # extract true[2]
    y_pred = data["y_pred"].values  # extract pred[2]
    pearson_r, r2, spearman_rho, mae, rmse, ccc = regression_metrics(y_true, y_pred)  # core metrics[4]
    slope, intercept = calibration_slope_intercept(y_true, y_pred)  # calibration slope/intercept[7]
    bias, loa_low, loa_high = bland_altman_stats(y_true, y_pred)  # Bland–Altman[6]
    ccc_lo, ccc_hi = bootstrap_ccc_ci(y_true, y_pred)  # CCC CI[4]
    print(f"{name}: N={len(y_true)} | r={pearson_r:.3f} r2={r2:.3f} rho={spearman_rho:.3f} MAE={mae:.3f} RMSE={rmse:.3f} CCC={ccc:.3f} [{ccc_lo:.3f},{ccc_hi:.3f}] | cal_slope={slope:.3f} cal_intercept={intercept:.3f} | bias={bias:.3f} LoA=({loa_low:.3f},{loa_high:.3f})")  # summary line [4]
    return dict(y_true=y_true, y_pred=y_pred, pearson_r=pearson_r, r2=r2, spearman_rho=spearman_rho, mae=mae, rmse=rmse, ccc=ccc, slope=slope, intercept=intercept, bias=bias, loa_low=loa_low, loa_high=loa_high)  # return dict[4]

in_stats = summarize_subset("inAD_true", in_ad)  # summarize in-AD[1]
out_stats = summarize_subset("inAD_false", out_ad)  # summarize out-AD[1]

# -----------------------------
# Reliability curves (binned) and ECE-like metric
# -----------------------------
def plot_reliability(ax, y_true, y_pred, title: str):
    # Plot mean observed vs mean predicted across quantile bins with identity line and ECE-like score[8]
    bin_pred, bin_true, bin_count, ece = reliability_curve(y_true, y_pred, n_bins=10)  # compute bins[8]
    ax.plot([min(bin_pred.min(), bin_true.min()), max(bin_pred.max(), bin_true.max())],
            [min(bin_pred.min(), bin_true.min()), max(bin_pred.max(), bin_true.max())],
            linestyle="--", color="gray", label="Ideal")  # identity line[8]
    ax.plot(bin_pred, bin_true, marker="o", label=f"Reliability (ECE≈{ece:.3f})")  # reliability curve[8]
    ax.set_xlabel("Mean predicted per bin")  # x label[8]
    ax.set_ylabel("Mean observed per bin")  # y label[8]
    ax.set_title(title)  # title[8]
    ax.legend()  # legend[8]

fig, axes = plt.subplots(1, 2, figsize=(12, 5), constrained_layout=True)  # canvas[2]
plot_reliability(axes, in_ad["y_true"].values, in_ad["y_pred"].values, "Reliability (in-AD)")  # in-AD plot[8]
plot_reliability(axes, out_ad["y_true"].values, out_ad["y_pred"].values, "Reliability (out-AD)")  # out-AD plot[5][8]
plt.savefig("reliability_curves.png", dpi=200)  # save figure[2]
plt.close(fig)  # close fig[2]

# -----------------------------
# Isotonic calibration (fit on half of in-AD; evaluate on the other half)
# -----------------------------
def train_eval_isotonic_on_inAD(in_df: pd.DataFrame):
    # Split in-AD into calibration and evaluation halves for post-hoc monotone recalibration of mean predictions[3]
    idx = np.arange(len(in_df))  # indices[2]
    rng.shuffle(idx)  # shuffle[2]
    mid = len(idx) // 2  # midpoint[2]
    cal_idx, eval_idx = idx[:mid], idx[mid:]  # split indices[2]
    cal = in_df.iloc[cal_idx]  # calibration half[3]
    eva = in_df.iloc[eval_idx]  # evaluation half[3]
    iso = fit_isotonic_calibrator(cal["y_true"].values, cal["y_pred"].values)  # fit isotonic[3]
    eva_cal = eva.copy()  # copy[2]
    eva_cal["y_pred_cal"] = iso.transform(eva_cal["y_pred"].values)  # apply calibrator[3]
    base = regression_metrics(eva_cal["y_true"].values, eva_cal["y_pred"].values)  # base metrics[4]
    cald = regression_metrics(eva_cal["y_true"].values, eva_cal["y_pred_cal"].values)  # calibrated metrics[4]
    print(f"Isotonic (eval half in-AD): base CCC={base[-1]:.3f} -> cal CCC={cald[-1]:.3f}")  # CCC change[4]
    return iso, eva_cal  # return calibrator and eval frame[3]

iso_model, eva_iso = train_eval_isotonic_on_inAD(in_ad)  # run isotonic calibration[3]

# -----------------------------
# Bland–Altman plots
# -----------------------------
def plot_bland_altman(ax, y_true, y_pred, title: str):
    # Plot Bland–Altman: differences vs means with bias and 95% LoA lines[6]
    y_true = np.asarray(y_true)  # array[6]
    y_pred = np.asarray(y_pred)  # array[6]
    mean_xy = (y_true + y_pred) / 2.0  # means[6]
    diff = y_pred - y_true  # differences[6]
    bias, loa_low, loa_high = bland_altman_stats(y_true, y_pred)  # stats[6]
    ax.scatter(mean_xy, diff, alpha=0.5, s=15)  # scatter[6]
    ax.axhline(bias, color="red", linestyle="-", label=f"Bias={bias:.3f}")  # bias line[6]
    ax.axhline(loa_low, color="orange", linestyle="--", label=f"LoA low={loa_low:.3f}")  # LoA low[6]
    ax.axhline(loa_high, color="orange", linestyle="--", label=f"LoA high={loa_high:.3f}")  # LoA high[6]
    ax.set_xlabel("Mean of true and pred")  # x label[6]
    ax.set_ylabel("Pred − True")  # y label[6]
    ax.set_title(title)  # title[6]
    ax.legend()  # legend[6]

fig, axes = plt.subplots(1, 2, figsize=(12, 5), constrained_layout=True)  # canvas[2]
plot_bland_altman(axes, in_ad["y_true"].values, in_ad["y_pred"].values, "Bland–Altman (in-AD)")  # in-AD BA[6]
plot_bland_altman(axes, out_ad["y_true"].values, out_ad["y_pred"].values, "Bland–Altman (out-AD)")  # out-AD BA[5][6]
plt.savefig("bland_altman.png", dpi=200)  # save fig[6]
plt.close(fig)  # close fig[2]

# -----------------------------
# Predicted vs observed with calibration line
# -----------------------------
def plot_pred_vs_obs(ax, y_true, y_pred, title: str):
    # Scatter of predictions vs observations with identity and fitted calibration line[7]
    slope, intercept = calibration_slope_intercept(y_true, y_pred)  # fit calibration[7]
    ax.scatter(y_true, y_pred, alpha=0.5, s=15)  # scatter[2]
    mn, mx = float(min(y_true.min(), y_pred.min())), float(max(y_true.max(), y_pred.max()))  # range[2]
    ax.plot([mn, mx], [mn, mx], "--", color="gray", label="Ideal")  # identity[7]
    xx = np.linspace(mn, mx, 100)  # grid[2]
    yy = intercept + slope * xx  # fitted line[7]
    ax.plot(xx, yy, color="red", label=f"Cal line: y={intercept:.2f}+{slope:.2f}x")  # calibration line[7]
    ax.set_xlabel("Observed")  # x label[2]
    ax.set_ylabel("Predicted")  # y label[2]
    ax.set_title(title)  # title[2]
    ax.legend()  # legend[2]

fig, axes = plt.subplots(1, 2, figsize=(12, 5), constrained_layout=True)  # canvas[2]
plot_pred_vs_obs(axes, in_ad["y_true"].values, in_ad["y_pred"].values, "Pred vs Obs (in-AD)")  # in-AD[7]
plot_pred_vs_obs(axes, out_ad["y_true"].values, out_ad["y_pred"].values, "Pred vs Obs (out-AD)")  # out-AD[5][7]
plt.savefig("pred_vs_obs.png", dpi=200)  # save[2]
plt.close(fig)  # close[2]

# -----------------------------
# Residual diagnostics
# -----------------------------
def plot_residuals(axs, y_true, y_pred, title_prefix: str):
    # Residuals vs prediction and QQ-plot to assess heteroscedasticity and normality assumptions[2]
    res = y_pred - y_true  # residuals[2]
    axs.scatter(y_pred, res, alpha=0.5, s=15)  # residuals vs pred[2]
    axs.axhline(0, color="gray", linestyle="--")  # zero line[2]
    axs.set_xlabel("Predicted")  # x label[2]
    axs.set_ylabel("Residual (Pred−True)")  # y label[2]
    axs.set_title(f"{title_prefix}: Residuals vs Pred")  # title[2]
    stats.probplot(res, dist="norm", plot=axs)  # QQ-plot[5][2]
    axs.set_title(f"{title_prefix}: Residuals QQ")  # title[5][2]

fig, axs = plt.subplots(2, 2, figsize=(12, 8), constrained_layout=True)  # canvas[2]
plot_residuals(axs, in_ad["y_true"].values, in_ad["y_pred"].values, "in-AD")  # in-AD resid[2]
plot_residuals(axs, out_ad["y_true"].values, out_ad["y_pred"].values, "out-AD")  # out-AD resid[5][2]
plt.savefig("residual_diagnostics.png", dpi=200)  # save fig[2]
plt.close(fig)  # close[2]


# -----------------------------
# Conformal prediction intervals (split conformal on in-AD)
# -----------------------------
def split_conformal_intervals(y_true: np.ndarray, y_pred: np.ndarray, alpha: float = 0.1):
    # Split conformal uses a calibration set residual quantile to form prediction intervals with finite-sample coverage[9]
    idx = np.arange(len(y_true))  # indices[9]
    rng.shuffle(idx)  # shuffle[9]
    mid = len(idx) // 2  # split point[9]
    cal_idx, eval_idx = idx[:mid], idx[mid:]  # splits[9]
    cal_res = np.abs(y_true[cal_idx] - y_pred[cal_idx])  # calibration residuals[9]
    q = np.quantile(cal_res, 1 - alpha)  # conformal quantile[9]
    lo = y_pred[eval_idx] - q  # lower bound[9]
    hi = y_pred[eval_idx] + q  # upper bound[9]
    covered = ((y_true[eval_idx] >= lo) & (y_true[eval_idx] <= hi)).mean()  # empirical coverage[9]
    print(f"Split conformal (in-AD eval half): nominal={1-alpha:.2f}, empirical={covered:.2f}, q={q:.3f}")  # report[9]
    return q, covered  # return quantile and coverage[9]

_ = split_conformal_intervals(in_ad["y_true"].values, in_ad["y_pred"].values, alpha=0.1)  # 90% intervals[9]

# -----------------------------
# Save a compact metrics table
# -----------------------------
summary_rows = []  # init list[2]
for name, dat in [("inAD_true", in_ad), ("inAD_false", out_ad)]:  # loop subsets[1]
    y_true = dat["y_true"].values  # true[2]
    y_pred = dat["y_pred"].values  # pred[2]
    pearson_r, r2, spearman_rho, mae, rmse, ccc = regression_metrics(y_true, y_pred)  # metrics[4]
    slope, intercept = calibration_slope_intercept(y_true, y_pred)  # calibration[7]
    bias, loa_low, loa_high = bland_altman_stats(y_true, y_pred)  # BA[6]
    row = dict(subset=name, N=len(y_true), pearson_r=pearson_r, r2=r2, spearman_rho=spearman_rho,
               MAE=mae, RMSE=rmse, CCC=ccc, cal_slope=slope, cal_intercept=intercept,
               bias=bias, loa_low=loa_low, loa_high=loa_high)  # row dict[4]
    summary_rows.append(row)  # append[2]
summary_df = pd.DataFrame(summary_rows)  # to DataFrame[2]
summary_df.to_csv("external_validation_summary.csv", index=False)  # save CSV[2]
print("Saved external_validation_summary.csv, reliability_curves.png, bland_altman.png, pred_vs_obs.png, residual_diagnostics.png")  # notify[2]

# -----------------------------
# Guidance (read this cell’s comments)
# -----------------------------
# Interpretation tips:
# - CCC summarizes both precision and accuracy; track its CI via bootstrap to judge whether 0.72 is materially below target thresholds.[4]
# - Bland–Altman bias and limits reveal systematic offsets and error spread across the range; wide LoA or trend vs mean suggests miscalibration or heteroscedasticity.[6]
# - Calibration slope <1 indicates overfitting (predictions too extreme), >1 underfitting; intercept captures calibration‑in‑the‑large (global bias).[7]
# - Reliability curve (binned mean observed vs mean predicted) should track the identity; ECE-like gap quantifies average deviation.[8]
# - Isotonic calibration can correct monotone bias without changing ranking; evaluate on a holdout split to avoid overfitting the mapping.[3]
# - Add uncertainty via split conformal intervals using calibration residual quantiles to get distribution‑free coverage on new points.[9]
# - Keep external test untouched for final reporting per OECD QSAR validation best practices and AD transparency.[1]


KeyError: 'y_true'

In [11]:
# === External validation diagnostics with specified columns: pre_reg_affinity (pred), dock_true_affinity (true), inAD_final (AD) ===[1]
# This cell computes CCC, correlation metrics, calibration line, reliability curves, Bland–Altman plots, residual diagnostics, isotonic calibration demo, split conformal intervals, and saves a summary CSV and PNG figures.[2][3][4][5][6]

# -----------------------------
# Imports
# -----------------------------
import os  # filesystem utilities for file checks and saving outputs[1]
import numpy as np  # numerical arrays and vectorized math[1]
import pandas as pd  # data loading and DataFrame manipulation[1]
from scipy import stats  # statistical functions including Pearson/Spearman and QQ-plot[1]
from sklearn.isotonic import IsotonicRegression  # monotone calibrator for regression outputs[3]
from sklearn.linear_model import LinearRegression  # ordinary least squares for calibration line[1]
import matplotlib.pyplot as plt  # plotting and saving figures to files[5]

# -----------------------------
# Configuration
# -----------------------------
INPUT_CSV = "df_full_wo_features_with_ev.csv"  # path to your CSV containing the specified columns[1]
RANDOM_SEED = 42  # random seed for reproducible splits and bootstraps[1]
N_BINS = 10  # number of quantile bins for reliability curve[1]
N_BOOT = 2000  # bootstrap samples for CCC confidence interval[2]
ALPHA_CONFORMAL = 0.1  # conformal alpha (e.g., 0.1 yields 90% nominal coverage)[6]
rng = np.random.default_rng(RANDOM_SEED)  # initialize NumPy random generator[1]

# -----------------------------
# Helpers: metrics and calibration
# -----------------------------
def concordance_correlation_coefficient(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    # Lin’s CCC: ρ_c = 2σ_xy / (σ_x^2 + σ_y^2 + (μ_x − μ_y)^2), measuring agreement in both scale and location[2]
    x, y = np.asarray(y_true), np.asarray(y_pred)  # convert to arrays for numeric operations[1]
    mx, my = x.mean(), y.mean()  # means of true and predicted values[2]
    vx, vy = x.var(ddof=1), y.var(ddof=1)  # sample variances of true and predicted values[2]
    sxy = np.cov(x, y, ddof=1)  # sample covariance between true and predicted values[7][2]
    return float((2 * sxy) / (vx + vy + (mx - my) ** 2))  # compute and return CCC value[2]

def bland_altman_stats(y_true: np.ndarray, y_pred: np.ndarray):
    # Bland–Altman bias and 95% limits of agreement: bias ± 1.96*SD(pred−true) for agreement assessment[4]
    d = np.asarray(y_pred) - np.asarray(y_true)  # per-sample differences (pred − true) for BA analysis[4]
    bias = d.mean()  # average difference as systematic bias estimate[4]
    sd = d.std(ddof=1)  # standard deviation of differences for LoA width[4]
    return float(bias), float(bias - 1.96 * sd), float(bias + 1.96 * sd)  # return bias and LoA bounds[4]

def calibration_slope_intercept(y_true: np.ndarray, y_pred: np.ndarray):
    # Fit OLS: y_true = intercept + slope * y_pred to quantify linear calibration and global bias[1]
    lr = LinearRegression()  # instantiate OLS regressor for calibration line fitting[1]
    yp = np.asarray(y_pred).reshape(-1, 1)  # shape predictions as a single regressor column[1]
    lr.fit(yp, np.asarray(y_true))  # fit regression line mapping predictions to observed values[1]
    return float(lr.coef_), float(lr.intercept_)  # return slope and intercept of calibration line[1]

def regression_metrics(y_true: np.ndarray, y_pred: np.ndarray):
    # Compute Pearson r, R^2, Spearman ρ, MAE, RMSE, and Lin’s CCC for comprehensive evaluation[2]
    pearson_r = stats.pearsonr(y_true, y_pred).statistic  # linear correlation between predictions and observations[1]
    r2 = pearson_r ** 2  # coefficient of determination from Pearson correlation in simple summary form[1]
    spearman_rho = stats.spearmanr(y_true, y_pred).correlation  # rank correlation for monotonic association[1]
    mae = float(np.mean(np.abs(y_pred - y_true)))  # mean absolute error as magnitude of typical error[1]
    rmse = float(np.sqrt(np.mean((y_pred - y_true) ** 2)))  # root mean squared error sensitive to outliers[1]
    ccc = concordance_correlation_coefficient(y_true, y_pred)  # agreement metric combining precision and accuracy[2]
    return pearson_r, r2, spearman_rho, mae, rmse, ccc  # return metrics tuple for reporting[2]

def reliability_curve(y_true: np.ndarray, y_pred: np.ndarray, n_bins: int = 10):
    # Bin by predicted quantiles; compute mean predicted vs mean observed per bin for a regression calibration curve[1]
    order = np.argsort(y_pred)  # indices for sorting by predicted values to form equal-count bins[1]
    yt, yp = np.asarray(y_true)[order], np.asarray(y_pred)[order]  # sorted arrays aligned by predictions[1]
    bins = np.array_split(np.arange(len(yp)), n_bins)  # split into equal-size index bins for calibration plotting[1]
    bin_pred = np.array([yp[idx].mean() for idx in bins])  # mean predicted per bin on x-axis[1]
    bin_true = np.array([yt[idx].mean() for idx in bins])  # mean observed per bin on y-axis[1]
    bin_count = np.array([len(idx) for idx in bins])  # count per bin for weighting summary stats[1]
    ece = float(np.average(np.abs(bin_true - bin_pred), weights=bin_count / len(yp)))  # ECE-like average absolute calibration gap[1]
    return bin_pred, bin_true, bin_count, ece  # return arrays for plotting and summary ECE-like score[1]

def fit_isotonic_calibrator(y_true: np.ndarray, y_pred: np.ndarray):
    # Fit monotone mapping g so that calibrated predictions = g(y_pred), preserving ranking and correcting monotone bias[3]
    iso = IsotonicRegression(out_of_bounds="clip")  # isotonic regressor with clipping beyond fit range[3]
    iso.fit(y_pred, y_true)  # fit non-decreasing function to map predictions to observed scale[3]
    return iso  # return fitted calibrator for later transform on evaluation set[3]

def bootstrap_ccc_ci(y_true: np.ndarray, y_pred: np.ndarray, n_boot: int = 2000, alpha: float = 0.05):
    # Percentile bootstrap confidence interval for CCC to quantify uncertainty of agreement estimate[2]
    n = len(y_true)  # number of validation samples for bootstrap resampling[1]
    idx = rng.integers(0, n, size=(n_boot, n))  # bootstrap index matrix for resampling with replacement[1]
    ccc_samples = [concordance_correlation_coefficient(y_true[i], y_pred[i]) for i in idx]  # CCC across bootstrap replicates[2]
    lo, hi = float(np.quantile(ccc_samples, alpha / 2)), float(np.quantile(ccc_samples, 1 - alpha / 2))  # percentile CI bounds for CCC[2]
    return lo, hi  # return lower and upper bootstrap CI limits[2]

def split_conformal_intervals(y_true: np.ndarray, y_pred: np.ndarray, alpha: float = 0.1):
    # Split conformal intervals: use calibration residual quantile to produce distribution‑free prediction intervals on eval split[6]
    n = len(y_true)  # total number of samples for splitting[6]
    idx = np.arange(n)  # index vector to define a random split[6]
    rng.shuffle(idx)  # shuffle indices to randomize calibration/evaluation partition[6]
    mid = n // 2  # midpoint for halving the dataset into calibration and evaluation[6]
    cal_idx, eval_idx = idx[:mid], idx[mid:]  # index subsets for calibration and evaluation[6]
    q = float(np.quantile(np.abs(y_true[cal_idx] - y_pred[cal_idx]), 1 - alpha))  # residual quantile for nominal coverage 1−alpha[6]
    lo, hi = y_pred[eval_idx] - q, y_pred[eval_idx] + q  # interval bounds centered at point prediction with uniform width q[6]
    covered = float(((y_true[eval_idx] >= lo) & (y_true[eval_idx] <= hi)).mean())  # empirical coverage on evaluation split[6]
    return q, covered, eval_idx  # return width quantile, empirical coverage, and indices used for evaluation[6]

# -----------------------------
# Load data and extract required columns
# -----------------------------
if not os.path.exists(INPUT_CSV):  # ensure the CSV exists at the specified path before reading[1]
    raise FileNotFoundError(f"Missing file: {INPUT_CSV}")  # clear error if file is not found to prompt correction[1]

df = pd.read_csv(INPUT_CSV)  # load external validation data into a DataFrame from CSV[1]
required_cols = {"pre_reg_affinity", "dock_true_affinity", "inAD_final"}  # expected columns for predictions, truths, and AD flag[1]
missing = required_cols - set(df.columns)  # determine which required columns are absent in the provided file[1]
if missing:  # guard against schema mismatch to avoid cryptic errors down the line[1]
    raise ValueError(f"Missing required columns: {missing}")  # raise schema error with the missing names for quick fix[1]

y_pred_all = df["pre_reg_affinity"].values  # model predictions from the specified prediction column[1]
y_true_all = df["dock_true_affinity"].values  # observed docking affinities from the specified ground truth column[1]
mask_inAD = df["inAD_final"].astype(bool).values  # boolean mask for in-domain applicability based on provided flag[1]

# -----------------------------
# Build in-AD and out-AD subsets
# -----------------------------
in_ad_df = df.loc[mask_inAD].copy()  # DataFrame of in-AD samples for focused reliability assessment[1]
out_ad_df = df.loc[~mask_inAD].copy()  # DataFrame of out-AD samples to quantify degradation outside domain[1]

# -----------------------------
# Summary function for a subset
# -----------------------------
def summarize_subset(name: str, data: pd.DataFrame):
    # Compute metrics, calibration line, BA stats, and CCC bootstrap CI for a named subset[2]
    y_true = data["dock_true_affinity"].values  # extract observed values from the subset[1]
    y_pred = data["pre_reg_affinity"].values  # extract predicted values from the subset[1]
    pearson_r, r2, spearman_rho, mae, rmse, ccc = regression_metrics(y_true, y_pred)  # evaluate correlation, error, and agreement metrics[2]
    slope, intercept = calibration_slope_intercept(y_true, y_pred)  # fit calibration line: y_true ~ y_pred[1]
    bias, loa_low, loa_high = bland_altman_stats(y_true, y_pred)  # compute Bland–Altman bias and limits of agreement[4]
    ccc_lo, ccc_hi = bootstrap_ccc_ci(y_true, y_pred, n_boot=N_BOOT)  # bootstrap CI for CCC to express uncertainty[2]
    print(f"{name}: N={len(y_true)} | r={pearson_r:.6f} r2={r2:.6f} rho={spearman_rho:.6f} MAE={mae:.6f} RMSE={rmse:.6f} CCC={ccc:.6f} [{ccc_lo:.6f},{ccc_hi:.6f}] | cal_slope={slope:.6f} cal_intercept={intercept:.6f} | bias={bias:.6f} LoA=({loa_low:.6f},{loa_high:.6f})")  # single-line summary for logging [2]
    return dict(y_true=y_true, y_pred=y_pred, r=pearson_r, r2=r2, rho=spearman_rho, MAE=mae, RMSE=rmse, CCC=ccc, cal_slope=slope, cal_intercept=intercept, bias=bias, loa_low=loa_low, loa_high=loa_high)  # return dictionary of metrics for CSV[2]

in_stats = summarize_subset("inAD_true", in_ad_df)  # summarize in-AD subset metrics for diagnostics and reporting[2]
out_stats = summarize_subset("inAD_false", out_ad_df)  # summarize out-AD subset metrics to quantify extrapolation error[2]

# -----------------------------
# Reliability curves (regression calibration)
# -----------------------------
def plot_reliability(ax, y_true, y_pred, title: str):
    # Plot mean observed vs mean predicted across quantile bins with identity line and ECE-like deviation[1]
    bin_pred, bin_true, _, ece = reliability_curve(y_true, y_pred, n_bins=N_BINS)  # compute binned calibration statistics[1]
    xy_min = float(min(bin_pred.min(), bin_true.min()))  # lower bound for identity line spanning bin means[1]
    xy_max = float(max(bin_pred.max(), bin_true.max()))  # upper bound for identity line spanning bin means[1]
    ax.plot([xy_min, xy_max], [xy_min, xy_max], "--", color="gray", label="Ideal")  # 45° identity for perfect calibration reference[1]
    ax.plot(bin_pred, bin_true, marker="o", label=f"Reliability (ECE≈{ece:.3f})")  # empirical calibration curve across bins[1]
    ax.set_xlabel("Mean predicted per bin")  # label x-axis as average prediction within bins[1]
    ax.set_ylabel("Mean observed per bin")  # label y-axis as average observation within bins[1]
    ax.set_title(title)  # set plot title identifying subset context[1]
    ax.legend()  # show legend including ECE-like score[1]

fig, axes = plt.subplots(1, 2, figsize=(12, 5), constrained_layout=True)  # create side-by-side reliability plots for in/out AD[5]
plot_reliability(axes, in_ad_df["dock_true_affinity"].values, in_ad_df["pre_reg_affinity"].values, "Reliability (in-AD)")  # plot in-AD reliability curve[1]
plot_reliability(axes, out_ad_df["dock_true_affinity"].values, out_ad_df["pre_reg_affinity"].values, "Reliability (out-AD)")  # plot out-AD reliability curve[7][1]
plt.savefig("reliability_curves.png", dpi=200)  # save reliability curves image to file for review and reporting[5]
plt.close(fig)  # close figure to free memory in long sessions[5]

# -----------------------------
# Predicted vs observed with calibration line
# -----------------------------
def plot_pred_vs_obs(ax, y_true, y_pred, title: str):
    # Scatter observed vs predicted with 45° identity and OLS calibration line overlay[1]
    slope, intercept = calibration_slope_intercept(y_true, y_pred)  # compute linear calibration parameters[1]
    ax.scatter(y_true, y_pred, alpha=0.5, s=15)  # scatter points of observed vs predicted values[5]
    mn, mx = float(min(y_true.min(), y_pred.min())), float(max(y_true.max(), y_pred.max()))  # determine axis limits from data[1]
    ax.plot([mn, mx], [mn, mx], "--", color="gray", label="Ideal")  # identity line for perfect calibration reference[1]
    xx = np.linspace(mn, mx, 100)  # grid of x-values for drawing calibration line[1]
    yy = intercept + slope * xx  # fitted calibration line values across the grid[1]
    ax.plot(xx, yy, color="red", label=f"Cal line: y={intercept:.2f}+{slope:.2f}x")  # overlay calibration line on scatter[1]
    ax.set_xlabel("Observed")  # x-axis label for observed values[5]
    ax.set_ylabel("Predicted")  # y-axis label for predicted values[5]
    ax.set_title(title)  # set title identifying subset[5]
    ax.legend()  # show legend for identity and calibration lines[5]

fig, axes = plt.subplots(1, 2, figsize=(12, 5), constrained_layout=True)  # set up figure for in/out AD scatter plots[5]
plot_pred_vs_obs(axes, in_ad_df["dock_true_affinity"].values, in_ad_df["pre_reg_affinity"].values, "Pred vs Obs (in-AD)")  # in-AD scatter and calibration line[1]
plot_pred_vs_obs(axes, out_ad_df["dock_true_affinity"].values, out_ad_df["pre_reg_affinity"].values, "Pred vs Obs (out-AD)")  # out-AD scatter and calibration line[7][1]
plt.savefig("pred_vs_obs.png", dpi=200)  # save predicted vs observed figures to PNG[5]
plt.close(fig)  # close figure after saving[5]

# -----------------------------
# Bland–Altman diagnostic plots
# -----------------------------
def plot_bland_altman(ax, y_true, y_pred, title: str):
    # Difference vs mean scatter with bias and 95% LoA lines to assess agreement across the range[4]
    mean_xy = (np.asarray(y_true) + np.asarray(y_pred)) / 2.0  # per-point means for x-axis[4]
    diff = np.asarray(y_pred) - np.asarray(y_true)  # per-point differences for y-axis[4]
    bias, lo, hi = bland_altman_stats(y_true, y_pred)  # compute bias and LoA bounds for annotation[4]
    ax.scatter(mean_xy, diff, alpha=0.5, s=15)  # BA scatter plot of differences against means[5]
    ax.axhline(bias, color="red", linestyle="-", label=f"Bias={bias:.3f}")  # horizontal line at mean difference (bias)[4]
    ax.axhline(lo, color="orange", linestyle="--", label=f"LoA low={lo:.3f}")  # lower limit of agreement line[4]
    ax.axhline(hi, color="orange", linestyle="--", label=f"LoA high={hi:.3f}")  # upper limit of agreement line[4]
    ax.set_xlabel("Mean of true and pred")  # label x-axis for BA plot[5]
    ax.set_ylabel("Pred − True")  # label y-axis for BA plot[5]
    ax.set_title(title)  # set BA plot title for subset[5]
    ax.legend()  # show legend for bias and LoA annotations[5]

fig, axes = plt.subplots(1, 2, figsize=(12, 5), constrained_layout=True)  # set up Bland–Altman plots for both subsets[5]
plot_bland_altman(axes, in_ad_df["dock_true_affinity"].values, in_ad_df["pre_reg_affinity"].values, "Bland–Altman (in-AD)")  # BA plot in-AD[4]
plot_bland_altman(axes, out_ad_df["dock_true_affinity"].values, out_ad_df["pre_reg_affinity"].values, "Bland–Altman (out-AD)")  # BA plot out-AD[7][4]
plt.savefig("bland_altman.png", dpi=200)  # save BA plots to PNG file[5]
plt.close(fig)  # close BA figure to release resources[5]

# -----------------------------
# Residual diagnostics: residuals vs predicted and QQ plots
# -----------------------------
def plot_residuals(axs, y_true, y_pred, title_prefix: str):
    # Residual scatter and QQ-plot to check heteroscedasticity and residual distribution shape[1]
    res = np.asarray(y_pred) - np.asarray(y_true)  # compute residuals as pred − true for diagnostic plots[1]
    axs.scatter(y_pred, res, alpha=0.5, s=15)  # plot residuals vs predicted to reveal variance structure[5]
    axs.axhline(0, color="gray", linestyle="--")  # reference zero line for residual symmetry check[5]
    axs.set_xlabel("Predicted")  # label x-axis as predicted values[5]
    axs.set_ylabel("Residual (Pred−True)")  # label y-axis as residual values[5]
    axs.set_title(f"{title_prefix}: Residuals vs Pred")  # title for residual scatter plot[5]
    stats.probplot(res, dist="norm", plot=axs)  # QQ-plot against normal distribution for residuals[7][1]
    axs.set_title(f"{title_prefix}: Residuals QQ")  # title for residual QQ-plot[7][5]

fig, axs = plt.subplots(2, 2, figsize=(12, 8), constrained_layout=True)  # grid for residual diagnostics for both subsets[5]
plot_residuals(axs, in_ad_df["dock_true_affinity"].values, in_ad_df["pre_reg_affinity"].values, "in-AD")  # residual diagnostics in-AD[1]
plot_residuals(axs, out_ad_df["dock_true_affinity"].values, out_ad_df["pre_reg_affinity"].values, "out-AD")  # residual diagnostics out-AD[7][1]
plt.savefig("residual_diagnostics.png", dpi=200)  # save residual diagnostic plots to file[5]
plt.close(fig)  # close residual figure after saving[5]

# -----------------------------
# Optional: per-scaffold error summary if a 'scaffold' column exists
# -----------------------------
if "scaffold" in df.columns:  # only compute scaffold-level summary when scaffold labels are available[1]
    err_df = in_ad_df.assign(abs_err=lambda d: np.abs(d["pre_reg_affinity"] - d["dock_true_affinity"]))  # per-sample absolute errors in in-AD[1]
    per_scf = err_df.groupby("scaffold")["abs_err"].agg(["count", "mean"]).sort_values("mean", ascending=False)  # scaffold counts and mean abs error[1]
    per_scf.to_csv("per_scaffold_error_inAD.csv", index=True)  # write scaffold error summary for chemistry review[1]

# -----------------------------
# Isotonic calibration demo on in-AD (calibrate half, evaluate half)
# -----------------------------
def isotonic_demo(in_df: pd.DataFrame):
    # Demonstrate post‑hoc monotone recalibration of predictions and report CCC change on held‑out half[3]
    idx = np.arange(len(in_df))  # index vector for random split of in-AD data[1]
    rng.shuffle(idx)  # shuffle to avoid ordering effects in split[1]
    mid = len(idx) // 2  # midpoint for 50/50 calibration/evaluation split[6]
    cal, eva = in_df.iloc[idx[:mid]], in_df.iloc[idx[mid:]]  # split into calibration and evaluation halves[6]
    iso = fit_isotonic_calibrator(cal["dock_true_affinity"].values, cal["pre_reg_affinity"].values)  # fit isotonic mapping on calibration half[3]
    eva_cal = eva.copy()  # copy evaluation half to add calibrated predictions[1]
    eva_cal["y_pred_cal"] = iso.transform(eva_cal["pre_reg_affinity"].values)  # apply calibrator to evaluation predictions[3]
    base_ccc = concordance_correlation_coefficient(eva_cal["dock_true_affinity"].values, eva_cal["pre_reg_affinity"].values)  # CCC before calibration[2]
    cal_ccc = concordance_correlation_coefficient(eva_cal["dock_true_affinity"].values, eva_cal["y_pred_cal"].values)  # CCC after isotonic calibration[2]
    print(f"Isotonic (eval half in-AD): base CCC={base_ccc:.3f} -> cal CCC={cal_ccc:.3f}")  # print CCC change to judge effect[2]
    return iso, eva_cal  # return fitted calibrator and evaluation DataFrame with calibrated predictions[3]

iso_model, eva_iso = isotonic_demo(in_ad_df)  # run isotonic calibration demo for in-AD subset[3]

# -----------------------------
# Split conformal intervals on in-AD to quantify uncertainty
# -----------------------------
q90, covg, eval_idx = split_conformal_intervals(in_ad_df["dock_true_affinity"].values, in_ad_df["pre_reg_affinity"].values, alpha=ALPHA_CONFORMAL)  # compute conformal width and coverage[6]
print(f"Split conformal (in-AD eval half): nominal={1-ALPHA_CONFORMAL:.2f}, empirical={covg:.2f}, q={q90:.4f}")  # report nominal vs empirical coverage and width[6]

# -----------------------------
# Save compact metrics table
# -----------------------------
summary_df = pd.DataFrame([
    dict(subset="inAD_true", N=len(in_ad_df), **{k: v for k, v in in_stats.items() if k not in ("y_true", "y_pred")}),  # in-AD metrics row for CSV[2]
    dict(subset="inAD_false", N=len(out_ad_df), **{k: v for k, v in out_stats.items() if k not in ("y_true", "y_pred")}),  # out-AD metrics row for CSV[2]
])  # aggregate subset summaries into a single DataFrame for export[1]
summary_df.to_csv("external_validation_summary.csv", index=False)  # write summary metrics to CSV for records[1]
print("Saved: reliability_curves.png, pred_vs_obs.png, bland_altman.png, residual_diagnostics.png, external_validation_summary.csv")  # list generated artifacts for quick access[5]


TypeError: only length-1 arrays can be converted to Python scalars